<a href="https://colab.research.google.com/github/astridcvr/daily-ml-practice/blob/main/day_18_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# Day 18 - Hyperparameter Tuning / Ajustement d'hyperparamètres
# GridSearchCV for Ridge Regression
# =========================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# =========================================================
# 1. Load datasets
# =========================================================

print("Loading datasets from GitHub...\n")

X_train = pd.read_csv("https://raw.githubusercontent.com/astridcvr/daily-ml-practice/main/data/X_train.csv")
X_test  = pd.read_csv("https://raw.githubusercontent.com/astridcvr/daily-ml-practice/main/data/X_test.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/astridcvr/daily-ml-practice/main/data/y_train.csv")
y_test  = pd.read_csv("https://raw.githubusercontent.com/astridcvr/daily-ml-practice/main/data/y_test.csv")

print("Shapes:")
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

# =========================================================
# 2. Keep numeric columns only (same as Day 16–17)
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns
X_train_num = X_train[numeric_cols]
X_test_num  = X_test[numeric_cols]

print("\nNumeric dataset:")
print("X_train_num:", X_train_num.shape, "X_test_num:", X_test_num.shape)

# =========================================================
# 3. Define model + hyperparameter grid
# =========================================================

ridge = Ridge()

param_grid = {
    "alpha": [0.01, 0.1, 1, 5, 10, 50, 100],
    "fit_intercept": [True, False]
}

grid_search = GridSearchCV(
    ridge,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# =========================================================
# 4. Run GridSearchCV
# =========================================================

print("\nRunning GridSearchCV...\n")
grid_search.fit(X_train_num, y_train.values.ravel())

print("Best Parameters:", grid_search.best_params_)
print("Best MSE (negative):", grid_search.best_score_)

# Convert negative MSE to positive
best_mse = -grid_search.best_score_
print("Best MSE:", best_mse)

# =========================================================
# 5. Evaluate best model on test set
# =========================================================

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test_num)

test_mse = mean_squared_error(y_test, y_pred)
test_r2  = r2_score(y_test, y_pred)

print("\nTest set results:")
print("MSE:", test_mse)
print("R² :", test_r2)

# =========================================================
# 6. Save results locally
# =========================================================

results_df = pd.DataFrame({
    "Metric": ["Best Alpha", "Train_MSE", "Test_MSE", "Test_R2"],
    "Value": [
        grid_search.best_params_["alpha"],
        best_mse,
        test_mse,
        test_r2
    ]
})

output_path = "day_18_tuning_results.csv"
results_df.to_csv(output_path, index=False)

print("\nTuning results saved locally:", output_path)
print(results_df)


Loading datasets from GitHub...

Shapes:
X_train: (2258, 48) X_test: (565, 48)
y_train: (2258, 1) y_test: (565, 1)

Numeric dataset:
X_train_num: (2258, 8) X_test_num: (565, 8)

Running GridSearchCV...

Best Parameters: {'alpha': 100, 'fit_intercept': False}
Best MSE (negative): -610430.5203699168
Best MSE: 610430.5203699168

Test set results:
MSE: 1036353.8330113114
R² : 0.7625704668185129

Tuning results saved locally: day_18_tuning_results.csv
       Metric         Value
0  Best Alpha  1.000000e+02
1   Train_MSE  6.104305e+05
2    Test_MSE  1.036354e+06
3     Test_R2  7.625705e-01
